# Creating Dataset Splits

For AI/ML applications, datasets are typically split into training, validation, and test sets. These splits are used to evaluate the generalization performance of models and to prevent overfitting. However, the way these splits are created can vary between datasets or when done on the fly, making it challenging to replicate the behaviour across model training and evaluation runs.

For more consistent and reproducible results, the dataset should be split deterministicly while accommodating for different split strategies.

A counterintuitive example: the random split. Even for the random split, one could create such a split randomly, then share their split with others so that everyone has the same random split.

This notebook demonstrates how to create deterministic dataset splits using ProteinGym's splitting functionality. We'll cover:

1. **Random splits** - for general train/validation/test divisions
2. **K-fold splits** - for cross-validation workflows
3. **Predefined splits** - for incorporating splits from outside proteingym
4. **Archiving splits** - for sharing and reproducibility

## Core concepts:

### Dataset slice

The term "dataset slice" refers to the accessors that create a  subset of a
dataset. Slice is a common term in programming to create a subset of a data
structure, for example 
[array slicing](https://en.wikipedia.org/wiki/Array_slicing) or search for
"slice" together with your programming language of choice. 

Note that "slice" is used both as a verb 
([function](https://docs.python.org/3/library/functions.html#slice))
and a noun ([object](https://docs.python.org/3/glossary.html#term-slice)). Also
see 
[slicings in Python](https://docs.python.org/3/reference/expressions.html#slicings).

For the protein gym data model, a dataset slice is a
collection of accessors for a dataset to select specific (subsets of) assays,
sequences, structures, or MSAs. 

### Dataset operators

The introduction of splits introduces the notion for dataset operators for
communicating about the relationships between datasets, dataset splits, and
subsets of datasets (other term for splits).

For example, when splitting a dataset, for each split the following is true:

```
dataset contains split
split is contained in dataset
```

> See Wikipedia on this [subsets](https://en.wikipedia.org/wiki/Set_(mathematics)#Subsets)

When splitting a dataset into two, the following operations hold true:

```
split_1 union split_2 equals dataset
split_1 intersection split_2 equals empty set
dataset difference split_1 equals split_2
```

The difference between a dataset split and subset is that splits are always
[disjoint](https://en.wikipedia.org/wiki/Disjoint_sets) while subsets may
overlap.

```
split_1 intersection split_2 equals empty set        # Always disjoint
subset_1 intersection subset_2 not equals empty set  # May overlap
```

For the protein gym data model, this implies that a dataset
split needs to account for the interdependencies between the protein data types.
For example, the sequences split from an assay need to be split from the
list of sequences list.

## Loading the Dataset

Let's start by loading our example dataset:

In [1]:
from pathlib import Path
from proteingym.base import Dataset, Manifest
from proteingym.base.splits import RandomSplitter, KFoldSplitter

# Load the NEIME 2019 dataset
manifest_path = Path("../example_data/neime_2019.toml")
manifest = Manifest.from_path(manifest_path)
dataset = Dataset.from_manifest(manifest)

print(f"Loaded dataset: {dataset.name}")
print(f"Number of assay records: {len(dataset.assays[0].records)}")

Loaded dataset: NEIME_2019
Number of assay records: 922


## ProteinGym splitting patterns

In ProteinGym we always perform splitting in a two step approach. In the first step we initialize the split with the specific settings for the split, while in the second step we split the specific dataset. 

For example:

In [2]:
my_split_class = RandomSplitter(fractions=[0.8, 0.1, 0.1]) #80% train, 10% validation, 10% test
subsets = my_split_class.split(dataset=dataset)

subsets

Subsets(dataset=Dataset(
	name='NEIME_2019',
	description: The NEIME Kennouche 2019 (UniProt id: A0A1I9GEU1) datase,
	contents:
		assays: 1,
		assays_raw: 0,
		sequences: 1,
		structures: 1,
		msas: 1,
		assay_variables: 2,
		publication: Publication(
	title: Deep mutational scanning of the
                    Neisseri..., 
	author: Kennouche, Paul and Charles‐Orszag, Arthur and Nishiguchi, D..., 
	journal: The EMBO Journal, 
	volume: 38, 
	number: 22, 
	year: 2019, 
	pages: None, 
	doi: 10.15252/embj.2019102145, 
),
), slices=[DatasetSlice(assays=[AssaySlice(columns=None, records=[True, True, True, True, True, True, True, False, False, False, True, True, True, False, True, True, True, True, True, True, True, False, True, False, True, True, False, True, True, True, True, True, False, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, False, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True

This functionality facilitates research into different splitting algorithms. One can e.g. create two different sets of splits, and easily run proteingym-benchmark on them both to compare performance. E.g:

In [3]:
randomsplit = RandomSplitter(fractions=[0.8, 0.1, 0.1])
my_list_of_datasets = [dataset, dataset] #duplication of the NEIME dataset for illustrative purposes

for dataset in my_list_of_datasets:
    split_dataset = randomsplit.split(dataset=dataset)
    split_dataset.dump() # NOTE: This will create one .splits.pgdata file as its overwritten by the duplicate dataset

Then in proteingym-benchmark you can point your dataset folder to the location containing the newly split datasets.

## Random Splits

Random splits divide your data into training, validation, and test sets with specified proportions. The `RandomSplitter` can be made deterministic by passing a random seed.

### Three-way Split (Train/Validation/Test)

In [4]:
# Create a 80/10/10 train/validation/test split
random_splitter = RandomSplitter(fractions=[0.8, 0.1, 0.1], random_state=42)

print(f"Split proportions: {random_splitter.fractions}")

Split proportions: [0.8, 0.1, 0.1]


In [5]:
# Generate the split
split_superset = random_splitter.split(dataset=dataset)

print(f"Created {len(split_superset)} splits:")
split_names = ["Train", "Validation", "Test"]

for i, split_dataset in enumerate(split_superset):
    count = len(split_dataset.assays[0].records)
    total = len(dataset.assays[0].records)
    percentage = (count / total) * 100
    print(f"  {split_names[i]}: {count} samples ({percentage:.1f}%)")

Created 3 splits:
  Train: 738 samples (80.0%)
  Validation: 92 samples (10.0%)
  Test: 92 samples (10.0%)


### You can do any N-way split

In [6]:
n_way_split = RandomSplitter(fractions=[0.4, 0.3, 0.2, 0.1])
n_way_superset = n_way_split.split(dataset)

for split_dataset in n_way_superset:
    print(f"Split: {len(split_dataset.assays[0].records)} samples")

Split: 369 samples
Split: 277 samples
Split: 184 samples
Split: 92 samples


## K-Fold Cross-Validation Splits

K-fold splits divide data into k equal-sized folds for cross-validation. Each fold can serve as a test set while the remaining folds form the training set.

In [7]:
# Create 5-fold cross-validation splits
kfold_splitter = KFoldSplitter(n_splits=5, shuffle=True)

print(f"K-fold configuration:")
print(f"  Number of folds: {kfold_splitter.n_splits}")
print(f"  Shuffle data: {kfold_splitter.shuffle}")

K-fold configuration:
  Number of folds: 5
  Shuffle data: True


In [8]:
# Generate the k-fold splits
kfold_superset = kfold_splitter.split(dataset=dataset)

print(f"\nCreated {len(kfold_superset)} folds:")
total_samples = len(dataset.assays[0].records)

for i, fold_dataset in enumerate(kfold_superset):
    test_count = len(fold_dataset.assays[0].records)
    train_count = total_samples - test_count
    print(f"  Fold {i+1}: {train_count} train, {test_count} test samples")


Created 5 folds:
  Fold 1: 737 train, 185 test samples
  Fold 2: 737 train, 185 test samples
  Fold 3: 738 train, 184 test samples
  Fold 4: 738 train, 184 test samples
  Fold 5: 738 train, 184 test samples


## Working with Split Datasets

Each split returns a complete Dataset object with filtered data:

In [9]:
# Unpack the three-way split
train_dataset, val_dataset, test_dataset = tuple(split_superset)

print(f"Training dataset: {len(train_dataset.assays[0].records)} samples")
print(f"Validation dataset: {len(val_dataset.assays[0].records)} samples")
print(f"Test dataset: {len(test_dataset.assays[0].records)} samples")

# Each split is a complete Dataset object
print(f"\nTraining dataset type: {type(train_dataset)}")
print(f"Training dataset name: {train_dataset.name}")

Training dataset: 738 samples
Validation dataset: 92 samples
Test dataset: 92 samples

Training dataset type: <class 'proteingym.base.dataset.Dataset'>
Training dataset name: NEIME_2019


## Loading predefined splits

To use splits already defined elsewhere, you can declare columns containing split allocation in your input data as non-target (prediction) fields. These can then be used to create a split proteingym dataset.

To add a non-target column to an assay you can add the following section to the manifest:

<div style="width: 70%">
  <div class="alert alert-block alert-info">
[[ assays.non_targets ]] <br>
name = "MySplitColumn" <br>
alias = "splits" <br>
  </div>
</div>

These non-targets do not get further parsed into the assay, the will be e.g. excluded from the dataset.to_df() function. The purpose of non-targets is to facilitate the edge cases where you might want to access extra information on the assay, but is not needed as an input to your machine learning algorithm.


To split using predefined splits, we initialize the splitter similarly as before:

In [10]:
from proteingym.base.splits import PredefinedSplitter

predefined = PredefinedSplitter(
    split_column="split", # The aliased name of the column in your assay
    split_order=["train", "valid", "test"] # The ordering of the returned datasets
)

my_splits = predefined.split(dataset=dataset)

This will return you three splits, which are ordered according to the given split order. So e.g.

In [11]:
for dataset in my_splits:
    print(dataset.assays[0].records[0][3])

train
valid
test


The split_order is an essential component here, to facilitate the validation of the splits, there are a few checks performed under the hood of the splitter. 

- If the split column contains values that are not present in your split_order list, proteingym will throw an error informing you which values are missing
- Similarly, if the split_order contains values that are not present in your split column, proteingym will throw an error informing you which order value is missing.
- If one assay contains the split column but the other doesn't, we will return the splits based only on the assays containing the split column.

Thus there always needs to be a match between split values in your split column, and the ordering you provide to proteingym

## Archiving Splits for Reproducibility

You can save splits to share with collaborators or ensure reproducibility across experiments:

In [12]:
# Archive the random split
archive_path = split_superset.dump(path=Path("../example_data/"))
print(f"Split archived to: {archive_path}")
print(f"Archive size: {archive_path.stat().st_size / 1024:.1f} KB")

Split archived to: ../example_data/NEIME_2019.splits.pgdata
Archive size: 1327.0 KB


### Loading Archived Splits

In [13]:
# Load splits from archive
from proteingym.base import Subsets

loaded_splits = Subsets.from_path(archive_path)

print(f"Loaded splits from archive")
print(f"Number of splits: {len(loaded_splits)}")
print(f"Original dataset name: {loaded_splits.dataset.name}")

# Verify the splits are identical
original_train = tuple(split_superset)[0]
loaded_train = tuple(loaded_splits)[0]

original_count = len(original_train.assays[0].records)
loaded_count = len(loaded_train.assays[0].records)
print(f"\nSplits are identical: {original_count == loaded_count}")

Loaded splits from archive
Number of splits: 3
Original dataset name: NEIME_2019

Splits are identical: True


## Practical Usage Examples

### Example 1: Using Splits in ML Workflows

In [14]:
# Example: Extract training and test data for ML
train_dataset, val_dataset, test_dataset = tuple(split_superset)

# Extract sequences and targets from training set
train_records = train_dataset.assays[0].records
train_sequences = [record[0] for record in train_records]
train_targets = [record[1] for record in train_records]

print(f"Training set: {len(train_sequences)} sequences")
print(f"First training sequence: {train_sequences[0].value}...")
print(f"First training target: {train_targets[0]}")

# Extract test data
test_records = test_dataset.assays[0].records
test_sequences = [record[0] for record in test_records]
test_targets = [record[1] for record in test_records]

print(f"\nTest set: {len(test_sequences)} sequences")

Training set: 738 sequences
First training sequence: ITLIELMIVIAIVGILAAVALPAYQDYTARAQVSEAILLAEGQKSAVTEYYLNHGEWPGDNSSAGVATSADIKGKYVQSVTVANGVITAQMASSNVNNEIKSKKLSLWAKRQNGSVKWFCGQPVTRTTATATDVAAANGKTDDKINTKHLPSTCRDDSSAS...
First training target: -3.5980000000000003

Test set: 92 sequences


### Example 2: Cross-Validation Loop

In [15]:
# Example: Cross-validation workflow
cv_scores = []

for fold_idx, test_dataset in enumerate(kfold_superset):
    # Get test data
    test_records = test_dataset.assays[0].records
    n_test = len(test_records)
    
    # Calculate training size (total - test)
    total_samples = len(dataset.assays[0].records)
    n_train = total_samples - n_test
    
    # Simulate model training and evaluation
    # (In practice, you'd train your model here)
    simulated_score = 0.85 + (fold_idx * 0.02)  # Fake score for demo
    cv_scores.append(simulated_score)
    
    print(f"Fold {fold_idx + 1}: {n_train} train, {n_test} test → Score: {simulated_score:.3f}")

mean_score = sum(cv_scores) / len(cv_scores)
print(f"\nMean CV Score: {mean_score:.3f}")

Fold 1: 11 train, 185 test → Score: 0.850
Fold 2: 11 train, 185 test → Score: 0.870
Fold 3: 12 train, 184 test → Score: 0.890
Fold 4: 12 train, 184 test → Score: 0.910
Fold 5: 12 train, 184 test → Score: 0.930

Mean CV Score: 0.890


## Summary

In this notebook, we've learned how to:

1. **Create random splits** with specified proportions
2. **Generate k-fold splits** for cross-validation
3. **Archive and load splits** for reproducibility
4. **Extract data** from splits for ML workflows
5. **Apply best practices** for dataset splitting

The ProteinGym splitting functionality ensures your ML experiments are reproducible and comparable across different studies.

## Next Steps

Now you can:
- Apply these splitting strategies to your own datasets
- Integrate splits into your ML training pipelines
- Share standardized splits with collaborators
- Explore advanced splitting strategies for specific biological contexts